# 3D Reconstruction GAN — Volume Discriminator

This notebook implements your Phase 2 GAN strategy:

- Sparse GT slices → `Generator G` (pretrained `Fast2p5D`) → reconstructed slices
- `VolumeDiscriminator D` on **5-slice stacks** for z-axis consistency
- Two-phase training:
  1. Warm-up `D` with frozen `G` (5 epochs)
  2. Joint fine-tuning with `L_total = L_rec + λ_adv * L_adv`

`L_rec = MSE + (1 - SSIM) + FFT`, `L_adv` uses LSGAN (`MSE` on discriminator scores).

In [2]:
import random
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import nibabel as nib
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print('Device:', DEVICE)

CFG = {
    'data_root': '/Volumes/T7/ORACLE project/PKG-MU-Glioma-partial',
    'img_size': 256,
    'in_channels': 5,
    'context_depth': 5,
    'stack_len': 5,
    'max_timepoints': 10,
    'batch_size': 2,
    'num_workers': 0,
    'warmup_d_epochs': 3,
    'joint_epochs': 8,
    'lr_d': 1e-4,
    'lr_g': 1e-5,
    'lambda_adv': 0.01,
    'joint_d_step_every': 10,
    'epoch_save_dir': 'models/epoch_saves',
    'disc_ckpt': 'models/D_joint_epoch_022.pth',
    'gen_ckpt': 'models/G_joint_epoch_022.pth',
}
print(CFG)

Device: mps
{'data_root': '/Volumes/T7/ORACLE project/PKG-MU-Glioma-partial', 'img_size': 256, 'in_channels': 5, 'context_depth': 5, 'stack_len': 5, 'max_timepoints': 10, 'batch_size': 2, 'num_workers': 0, 'warmup_d_epochs': 3, 'joint_epochs': 8, 'lr_d': 0.0001, 'lr_g': 1e-05, 'lambda_adv': 0.01, 'joint_d_step_every': 10, 'epoch_save_dir': 'models/epoch_saves', 'disc_ckpt': 'models/D_joint_epoch_022.pth', 'gen_ckpt': 'models/G_joint_epoch_022.pth'}


In [3]:
def minmax_norm_2d(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = x.astype(np.float32)
    x_min = float(np.nanmin(x))
    x_max = float(np.nanmax(x))
    if x_max - x_min < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - x_min) / (x_max - x_min + eps)


def find_modality_file(timepoint_dir: Path, token: str) -> Path:
    files = sorted(timepoint_dir.glob('*.nii.gz'))
    if token == 'mask':
        candidates = [f for f in files if 'mask' in f.name.lower()]
    else:
        candidates = [f for f in files if token in f.name.lower()]
    if not candidates:
        raise FileNotFoundError(f'Could not find {token} in {timepoint_dir}')
    return candidates[0]


def load_timepoint_volumes(timepoint_dir: Path) -> Dict[str, np.ndarray]:
    paths = {
        't1n': find_modality_file(timepoint_dir, 't1n'),
        't1c': find_modality_file(timepoint_dir, 't1c'),
        't2w': find_modality_file(timepoint_dir, 't2w'),
        't2f': find_modality_file(timepoint_dir, 't2f'),
        'mask': find_modality_file(timepoint_dir, 'mask'),
    }
    vols = {k: nib.load(str(v)).get_fdata(dtype=np.float32) for k, v in paths.items()}
    base_shape = vols['t1n'].shape
    for k, v in vols.items():
        if v.shape != base_shape:
            raise ValueError(f'Shape mismatch in {timepoint_dir} for {k}: {v.shape} vs {base_shape}')
    return vols


def discover_timepoint_dirs(data_root: Path) -> List[Path]:
    timepoints: List[Path] = []
    for p in sorted(data_root.glob('PatientID_*')):
        if p.is_dir():
            for tp in sorted(p.glob('Timepoint_*')):
                if tp.is_dir():
                    timepoints.append(tp)
    if not timepoints:
        timepoints = sorted([p for p in data_root.rglob('Timepoint_*') if p.is_dir()])
    return timepoints


DATA_ROOT = Path(CFG['data_root'])
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Data root not found: {DATA_ROOT}')

ALL_TIMEPOINTS = discover_timepoint_dirs(DATA_ROOT)
if not ALL_TIMEPOINTS:
    raise RuntimeError('No timepoints found.')

TIMEPOINTS = ALL_TIMEPOINTS[:CFG['max_timepoints']]
print(f'Using {len(TIMEPOINTS)} timepoints out of {len(ALL_TIMEPOINTS)} discovered.')

Using 10 timepoints out of 15 discovered.


In [4]:
class ConvBlock2D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UNetDecoder2D(nn.Module):
    def __init__(self, in_ch: int = 128, out_ch: int = 1):
        super().__init__()
        self.enc1 = ConvBlock2D(in_ch, 128)
        self.pool = nn.MaxPool2d(2)
        self.enc2 = ConvBlock2D(128, 256)
        self.bottleneck = ConvBlock2D(256, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = ConvBlock2D(512, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = ConvBlock2D(256, 128)
        self.head = nn.Conv2d(128, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.head(d1))


class SliceCNN2D(nn.Module):
    def __init__(self, in_ch: int = 5, out_ch: int = 96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 48, 3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.Conv2d(48, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Fast2p5D(nn.Module):
    def __init__(self, in_ch: int = 5, feat_ch: int = 96, out_ch: int = 1):
        super().__init__()
        self.slice_cnn = SliceCNN2D(in_ch=in_ch, out_ch=feat_ch)
        self.depth_attn = nn.Sequential(
            nn.Linear(feat_ch, feat_ch // 2),
            nn.ReLU(inplace=True),
            nn.Linear(feat_ch // 2, 1),
        )
        self.decoder = UNetDecoder2D(in_ch=feat_ch, out_ch=out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, d, h, w = x.shape
        feats = []
        for di in range(d):
            feats.append(self.slice_cnn(x[:, :, di, :, :]))
        feats = torch.stack(feats, dim=1)  # [B,D,F,H,W]
        depth_desc = feats.mean(dim=(3, 4))
        logits = self.depth_attn(depth_desc).squeeze(-1)
        alpha = torch.softmax(logits, dim=1)[:, :, None, None, None]
        fused = (feats * alpha).sum(dim=1)
        return self.decoder(fused)


class VolumeDiscriminator(nn.Module):
    # Input: [B, 5, H, W] -> internally [B,1,5,H,W]
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(3, 4, 4), stride=(1, 2, 2), padding=(1, 1, 1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(32, 64, kernel_size=(3, 4, 4), stride=(1, 2, 2), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(64, 128, kernel_size=(3, 4, 4), stride=(1, 2, 2), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1)),
        )
        self.head = nn.Linear(128, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)  # [B,1,5,H,W]
        feat = self.net(x).flatten(1)
        return self.head(feat)


def load_generator(ckpt_path: Path) -> nn.Module:
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Generator checkpoint not found: {ckpt_path}')

    g = Fast2p5D(in_ch=CFG['in_channels'], feat_ch=96, out_ch=1).to(DEVICE)
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    state_dict = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in state_dict.items()}
    missing, unexpected = g.load_state_dict(state_dict, strict=False)
    print(f'Loaded G from {ckpt_path}')
    print(f'Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}')
    return g

def load_discriminator(ckpt_path: Path) -> nn.Module:
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Discriminator checkpoint not found: {ckpt_path}')

    d = VolumeDiscriminator().to(DEVICE)
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    state_dict = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in state_dict.items()}
    missing, unexpected = d.load_state_dict(state_dict, strict=False)
    print(f'Loaded D from {ckpt_path}')
    print(f'Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}')
    return d

G = load_generator(Path(CFG['gen_ckpt']))
D = load_discriminator(Path(CFG['disc_ckpt']))

print('G params:', f"{sum(p.numel() for p in G.parameters()):,}")
print('D params:', f"{sum(p.numel() for p in D.parameters()):,}")

Loaded G from models/G_joint_epoch_022.pth
Missing keys: 0 | Unexpected keys: 0
Loaded D from models/D_joint_epoch_022.pth
Missing keys: 0 | Unexpected keys: 0
G params: 7,603,186
D params: 493,793


In [5]:
def make_known_mask_half_alternating(z_count: int) -> np.ndarray:
    known = np.zeros(z_count, dtype=bool)
    known[::2] = True
    if not known.any():
        known[0] = True
    return known


def make_known_mask_n_uniform(z_count: int, n_keep: int = 50) -> np.ndarray:
    n_keep = int(max(1, min(n_keep, z_count)))
    idx = np.unique(np.linspace(0, z_count - 1, num=n_keep, dtype=int))
    known = np.zeros(z_count, dtype=bool)
    known[idx] = True
    return known


def make_known_mask_random_fraction(z_count: int, frac: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n_keep = int(max(1, min(z_count, round(z_count * frac))))
    idx = np.sort(rng.choice(np.arange(z_count), size=n_keep, replace=False))
    known = np.zeros(z_count, dtype=bool)
    known[idx] = True
    return known


def make_known_mask_center_block(z_count: int, n_keep: int = 25) -> np.ndarray:
    n_keep = int(max(1, min(n_keep, z_count)))
    start = max(0, (z_count - n_keep) // 2)
    known = np.zeros(z_count, dtype=bool)
    known[start:start + n_keep] = True
    return known


def build_sparse_known_masks(z_count: int) -> Dict[str, np.ndarray]:
    return {
        'A_half_alternating_all5': make_known_mask_half_alternating(z_count),
        'B_50_uniform_all5': make_known_mask_n_uniform(z_count, n_keep=50),
        'C_half_alternating_t1n_mask': make_known_mask_half_alternating(z_count),
        'D_random_30_all5': make_known_mask_random_fraction(z_count, frac=0.30, seed=SEED),
        'E_random_15_all5': make_known_mask_random_fraction(z_count, frac=0.15, seed=SEED + 1),
        'F_center_25_all5': make_known_mask_center_block(z_count, n_keep=25),
    }


def init_t1n_state_from_known(t1n_gt: np.ndarray, known_mask: np.ndarray) -> np.ndarray:
    h, w, z_count = t1n_gt.shape
    known_ids = np.where(known_mask)[0]
    if len(known_ids) == 0:
        raise ValueError('known_mask has no known slices')

    state = np.zeros_like(t1n_gt, dtype=np.float32)
    for z in range(z_count):
        if known_mask[z]:
            state[:, :, z] = t1n_gt[:, :, z]
        else:
            nearest = int(known_ids[np.argmin(np.abs(known_ids - z))])
            state[:, :, z] = t1n_gt[:, :, nearest]
    return state


def build_context_slice(
    vols: Dict[str, np.ndarray],
    t1n_state: np.ndarray,
    known_mask: np.ndarray,
    z_id: int,
    img_size: int,
    input_mode: str,
    use_pred_for_all_channels: bool = True,
) -> np.ndarray:
    z_max = vols['t1n'].shape[2] - 1
    z = int(np.clip(z_id, 0, z_max))

    mask_base = cv2.resize((vols['mask'][:, :, z] > 0).astype(np.float32), (img_size, img_size), interpolation=cv2.INTER_LINEAR)

    if known_mask[z]:
        t1n_base = cv2.resize(minmax_norm_2d(vols['t1n'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        if input_mode == 't1n_mask':
            zero = np.zeros_like(t1n_base, dtype=np.float32)
            c_t1n, c_t1c, c_t2w, c_t2f, c_msk = t1n_base, zero, zero, zero, mask_base
        else:
            c_t1n = t1n_base
            c_t1c = cv2.resize(minmax_norm_2d(vols['t1c'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
            c_t2w = cv2.resize(minmax_norm_2d(vols['t2w'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
            c_t2f = cv2.resize(minmax_norm_2d(vols['t2f'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
            c_msk = mask_base
    else:
        pred_base = cv2.resize(minmax_norm_2d(t1n_state[:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        if input_mode == 't1n_mask':
            zero = np.zeros_like(pred_base, dtype=np.float32)
            c_t1n, c_t1c, c_t2w, c_t2f, c_msk = pred_base, zero, zero, zero, mask_base
        else:
            if use_pred_for_all_channels:
                c_t1n, c_t1c, c_t2w, c_t2f, c_msk = pred_base, pred_base, pred_base, pred_base, pred_base
            else:
                c_t1n = pred_base
                c_t1c = cv2.resize(minmax_norm_2d(vols['t1c'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
                c_t2w = cv2.resize(minmax_norm_2d(vols['t2w'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
                c_t2f = cv2.resize(minmax_norm_2d(vols['t2f'][:, :, z]), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
                c_msk = mask_base

    return np.stack([c_t1n, c_t1c, c_t2w, c_t2f, c_msk], axis=0).astype(np.float32)  # [5,H,W]


class SparseVolumeStackDataset(Dataset):
    # returns:
    #   contexts: [T, C, D, H, W] where T=stack_len, C=5, D=5
    #   gt_stack: [T, H, W]
    #   variant_id: int
    def __init__(self, timepoints: List[Path], img_size: int = 256, stack_len: int = 5):
        self.img_size = img_size
        self.stack_len = stack_len

        self.variant_names = [
            'A_half_alternating_all5',
            'B_50_uniform_all5',
            'C_half_alternating_t1n_mask',
            'D_random_30_all5',
            'E_random_15_all5',
            'F_center_25_all5',
        ]
        self.variant_to_id = {v: i for i, v in enumerate(self.variant_names)}

        self.vol_cache: Dict[str, Dict[str, np.ndarray]] = {}
        self.records: List[Dict] = []

        for tp in timepoints:
            vols = load_timepoint_volumes(tp)
            self.vol_cache[str(tp)] = vols
            z_count = vols['t1n'].shape[2]
            if z_count < 10:
                continue

            max_z_start = z_count - (5 + stack_len)
            if max_z_start < 0:
                continue

            for variant_name in self.variant_names:
                for z_start in range(0, max_z_start + 1):
                    self.records.append({
                        'tp': str(tp),
                        'variant': variant_name,
                        'z_start': z_start,
                    })

        if not self.records:
            raise RuntimeError('SparseVolumeStackDataset is empty.')

        print(f'Dataset records: {len(self.records)}')

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        tp = rec['tp']
        variant_name = rec['variant']
        z_start = rec['z_start']

        vols = self.vol_cache[tp]
        z_count = vols['t1n'].shape[2]

        known_map = build_sparse_known_masks(z_count)
        known_mask = known_map[variant_name]

        input_mode = 't1n_mask' if 't1n_mask' in variant_name else 'all5'

        t1n_state = init_t1n_state_from_known(vols['t1n'], known_mask)

        contexts = []
        gt_stack = []

        # target slices t = z_start+5 ... z_start+9 (5 consecutive)
        for t in range(z_start + 5, z_start + 5 + self.stack_len):
            z_ids = [t - 5, t - 4, t - 3, t - 2, t - 1]
            depth_items = [
                build_context_slice(vols, t1n_state, known_mask, zi, self.img_size, input_mode)
                for zi in z_ids
            ]
            # [D,C,H,W] -> [C,D,H,W]
            x_dc = np.stack(depth_items, axis=0).astype(np.float32)
            x = np.transpose(x_dc, (1, 0, 2, 3)).astype(np.float32)
            contexts.append(x)

            gt_slice = cv2.resize(minmax_norm_2d(vols['t1n'][:, :, t]), (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            gt_stack.append(gt_slice.astype(np.float32))

        contexts = np.stack(contexts, axis=0).astype(np.float32)   # [T,C,D,H,W]
        gt_stack = np.stack(gt_stack, axis=0).astype(np.float32)   # [T,H,W]

        return {
            'contexts': torch.from_numpy(contexts),
            'gt_stack': torch.from_numpy(gt_stack),
            'variant_id': torch.tensor(self.variant_to_id[variant_name], dtype=torch.long),
        }


dataset = SparseVolumeStackDataset(TIMEPOINTS, img_size=CFG['img_size'], stack_len=CFG['stack_len'])

n_total = len(dataset)
n_train = int(0.8 * n_total)
indices = np.random.permutation(n_total)
train_idx = indices[:n_train]
val_idx = indices[n_train:]

train_ds = torch.utils.data.Subset(dataset, train_idx.tolist())
val_ds = torch.utils.data.Subset(dataset, val_idx.tolist())

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'])
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])

print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')

Dataset records: 8760
Train samples: 7008 | Val samples: 1752


In [6]:
def ssim_global(x: torch.Tensor, y: torch.Tensor, c1: float = 0.01**2, c2: float = 0.03**2) -> torch.Tensor:
    # x,y: [N,1,H,W]
    mu_x = x.mean(dim=(2, 3), keepdim=True)
    mu_y = y.mean(dim=(2, 3), keepdim=True)
    var_x = ((x - mu_x) ** 2).mean(dim=(2, 3), keepdim=True)
    var_y = ((y - mu_y) ** 2).mean(dim=(2, 3), keepdim=True)
    cov_xy = ((x - mu_x) * (y - mu_y)).mean(dim=(2, 3), keepdim=True)

    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * cov_xy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (var_x + var_y + c2))
    return ssim_map.mean()


def fft_l1_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # pred,target: [B,T,H,W]
    pred_fft = torch.fft.rfft2(pred, dim=(-2, -1))
    targ_fft = torch.fft.rfft2(target, dim=(-2, -1))
    return torch.mean(torch.abs(pred_fft - targ_fft))


def reconstruction_loss(pred_stack: torch.Tensor, gt_stack: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    # [B,T,H,W]
    mse = F.mse_loss(pred_stack, gt_stack)

    n = pred_stack.shape[0] * pred_stack.shape[1]
    pred_2d = pred_stack.reshape(n, 1, pred_stack.shape[-2], pred_stack.shape[-1])
    gt_2d = gt_stack.reshape(n, 1, gt_stack.shape[-2], gt_stack.shape[-1])
    ssim_term = 1.0 - ssim_global(pred_2d, gt_2d)

    fft_term = fft_l1_loss(pred_stack, gt_stack)
    total = mse + ssim_term + fft_term

    return total, {
        'mse': float(mse.item()),
        'ssim': float((1.0 - ssim_term).item()),
        'fft': float(fft_term.item()),
    }


def predict_stack_with_g(g_model: nn.Module, contexts: torch.Tensor) -> torch.Tensor:
    # contexts: [B,T,C,D,H,W] -> output [B,T,H,W]
    b, t, c, d, h, w = contexts.shape
    preds = []
    for ti in range(t):
        x = contexts[:, ti, :, :, :, :]   # [B,C,D,H,W]
        y = g_model(x).squeeze(1)          # [B,H,W]
        preds.append(y)
    return torch.stack(preds, dim=1)


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


opt_D = torch.optim.Adam(D.parameters(), lr=CFG['lr_d'], betas=(0.5, 0.999))
opt_G = torch.optim.Adam(G.parameters(), lr=CFG['lr_g'], betas=(0.5, 0.999))
scheduler_G = CosineAnnealingLR(opt_G, T_max=20, eta_min=1e-5)
scheduler_D = CosineAnnealingLR(opt_D, T_max=20, eta_min=2e-6)

In [ ]:
def run_epoch_d_only(loader: DataLoader) -> Dict[str, float]:
    D.train()
    G.eval()

    total_loss = 0.0
    total_real = 0.0
    total_fake = 0.0
    n_batches = 0

    for batch in loader:
        contexts = batch['contexts'].to(DEVICE)   # [B,T,C,D,H,W]
        gt_stack = batch['gt_stack'].to(DEVICE)   # [B,T,H,W]

        with torch.no_grad():
            pred_stack = predict_stack_with_g(G, contexts)

        real_score = D(gt_stack)
        fake_score = D(pred_stack.detach())

        ones = torch.ones_like(real_score)
        zeros = torch.zeros_like(fake_score)

        loss_d = F.mse_loss(real_score, ones) + F.mse_loss(fake_score, zeros)

        opt_D.zero_grad(set_to_none=True)
        loss_d.backward()
        opt_D.step()

        total_loss += float(loss_d.item())
        total_real += float(real_score.mean().item())
        total_fake += float(fake_score.mean().item())
        n_batches += 1

    return {
        'loss_d': total_loss / max(1, n_batches),
        'real_score': total_real / max(1, n_batches),
        'fake_score': total_fake / max(1, n_batches),
    }


def run_epoch_joint(loader: DataLoader, d_step_every: int = 10) -> Dict[str, float]:
    D.train()
    G.train()

    total_loss_d = 0.0
    total_loss_g = 0.0
    total_loss_rec = 0.0
    total_loss_adv = 0.0
    total_psnr = 0.0
    n_batches = 0
    n_d_updates = 0

    for i, batch in enumerate(loader):
        contexts = batch['contexts'].to(DEVICE)   # [B,T,C,D,H,W]
        gt_stack = batch['gt_stack'].to(DEVICE)   # [B,T,H,W]

        # --- G step: every batch ---
        pred_stack = predict_stack_with_g(G, contexts)
        fake_score_for_g = D(pred_stack)

        loss_rec, rec_parts = reconstruction_loss(pred_stack, gt_stack)
        loss_adv = F.mse_loss(fake_score_for_g, torch.ones_like(fake_score_for_g))
        loss_g = loss_rec + CFG['lambda_adv'] * loss_adv

        opt_G.zero_grad(set_to_none=True)
        loss_g.backward()
        opt_G.step()
        scheduler_G.step()

        # --- D step: only every N batches (after warmup) ---
        if i % d_step_every == 0:
            with torch.no_grad():
                pred_det = predict_stack_with_g(G, contexts)

            real_score = D(gt_stack)
            fake_score = D(pred_det)
            ones = torch.ones_like(real_score)
            zeros = torch.zeros_like(fake_score)

            loss_d = F.mse_loss(real_score, ones) + F.mse_loss(fake_score, zeros)
            opt_D.zero_grad(set_to_none=True)
            loss_d.backward()
            opt_D.step()
            scheduler_D.step()

            total_loss_d += float(loss_d.item())
            n_d_updates += 1

        mse = F.mse_loss(pred_stack, gt_stack).item()
        psnr = 10.0 * np.log10(1.0 / max(mse, 1e-12))

        total_loss_g += float(loss_g.item())
        total_loss_rec += float(loss_rec.item())
        total_loss_adv += float(loss_adv.item())
        total_real += float(real_score.mean().item())
        total_fake += float(fake_score.mean().item())
        total_psnr += float(psnr)
        n_batches += 1
    print(f"LR G: {scheduler_G.get_last_lr()[0]:.2e}")
    print(f"LR D: {scheduler_D.get_last_lr()[0]:.2e}")

    return {
        'loss_d': total_loss_d / max(1, n_d_updates),
        'loss_g': total_loss_g / max(1, n_batches),
        'loss_rec': total_loss_rec / max(1, n_batches),
        'loss_adv': total_loss_adv / max(1, n_batches),
        'psnr': total_psnr / max(1, n_batches),
        'd_updates': n_d_updates,
        'real_score': total_real / max(1, n_batches),
        'fake_score': total_fake / max(1, n_batches),
    }


def save_models_for_epoch(phase: str, epoch: int):
    save_dir = Path(CFG['epoch_save_dir'])
    save_dir.mkdir(parents=True, exist_ok=True)

    g_path = save_dir / f'G_{phase}_epoch_{epoch:03d}.pth'
    d_path = save_dir / f'D_{phase}_epoch_{epoch:03d}.pth'

    torch.save({'model_state_dict': G.state_dict(), 'cfg': CFG, 'phase': phase, 'epoch': epoch}, g_path)
    torch.save({'model_state_dict': D.state_dict(), 'cfg': CFG, 'phase': phase, 'epoch': epoch}, d_path)

    print(f'Saved G: {g_path}')
    print(f'Saved D: {d_path}')


# --------- Phase 2 training strategy ---------
# 1) Freeze G backbone, warm-up D (D updates every batch)
set_requires_grad(G, False)
set_requires_grad(D, True)

history = []
for ep in range(1, CFG['warmup_d_epochs'] + 1):
    tr = run_epoch_d_only(train_loader)
    va = run_epoch_d_only(val_loader)
    row = {
        'phase': 'warmup_d',
        'epoch': ep,
        'train_loss_d': tr['loss_d'],
        'val_loss_d': va['loss_d'],
        'val_real_score': va['real_score'],
        'val_fake_score': va['fake_score'],
    }
    history.append(row)
    print(
        f"[Warmup D {ep:02d}/{CFG['warmup_d_epochs']}] "
        f"train D {tr['loss_d']:.4f} | val D {va['loss_d']:.4f} | "
        f"val real {va['real_score']:.4f} | val fake {va['fake_score']:.4f}"
    )
    save_models_for_epoch('warmup', ep)

# 2) Unfreeze G and train jointly (D updates every 10 batches)
set_requires_grad(G, True)
set_requires_grad(D, True)

for ep in range(1, CFG['joint_epochs'] + 1):
    tr = run_epoch_joint(train_loader, d_step_every=CFG['joint_d_step_every'])
    va = run_epoch_joint(val_loader, d_step_every=CFG['joint_d_step_every'])
    row = {
        'phase': 'joint',
        'epoch': ep,
        'train_loss_d': tr['loss_d'],
        'train_loss_g': tr['loss_g'],
        'train_loss_rec': tr['loss_rec'],
        'train_loss_adv': tr['loss_adv'],
        'train_psnr': tr['psnr'],
        'train_d_updates': tr['d_updates'],
        'val_loss_d': va['loss_d'],
        'val_loss_g': va['loss_g'],
        'val_loss_rec': va['loss_rec'],
        'val_loss_adv': va['loss_adv'],
        'val_psnr': va['psnr'],
        'val_d_updates': va['d_updates'],
    }
    history.append(row)
    print(
        f"[Joint {ep:02d}/{CFG['joint_epochs']}] "
        f"train PSNR {tr['psnr']:.2f} dB | val PSNR {va['psnr']:.2f} dB | "
        f"train G {tr['loss_g']:.4f} | val G {va['loss_g']:.4f} | "
        f"D updates train/val: {tr['d_updates']}/{va['d_updates']}"
    )
    save_models_for_epoch('joint', ep)

In [ ]:
@torch.no_grad()
def evaluate_by_variant(loader: DataLoader) -> Dict[str, Dict[str, float]]:
    G.eval()
    D.eval()

    id_to_variant = {i: n for n, i in dataset.variant_to_id.items()}
    stats: Dict[str, Dict[str, List[float]]] = {}

    for batch in loader:
        contexts = batch['contexts'].to(DEVICE)
        gt_stack = batch['gt_stack'].to(DEVICE)
        variant_ids = batch['variant_id'].cpu().numpy().tolist()

        pred_stack = predict_stack_with_g(G, contexts)

        real_score = D(gt_stack).squeeze(1).cpu().numpy()
        fake_score = D(pred_stack).squeeze(1).cpu().numpy()

        mse_each = torch.mean((pred_stack - gt_stack) ** 2, dim=(1, 2, 3)).cpu().numpy()
        psnr_each = [10.0 * np.log10(1.0 / max(float(m), 1e-12)) for m in mse_each]

        for i, vid in enumerate(variant_ids):
            vname = id_to_variant[int(vid)]
            if vname not in stats:
                stats[vname] = {'psnr': [], 'real_score': [], 'fake_score': []}
            stats[vname]['psnr'].append(float(psnr_each[i]))
            stats[vname]['real_score'].append(float(real_score[i]))
            stats[vname]['fake_score'].append(float(fake_score[i]))

    out: Dict[str, Dict[str, float]] = {}
    for v, d in stats.items():
        out[v] = {
            'psnr': float(np.mean(d['psnr'])),
            'real_score': float(np.mean(d['real_score'])),
            'fake_score': float(np.mean(d['fake_score'])),
            'score_gap': float(np.mean(d['real_score']) - np.mean(d['fake_score'])),
        }
    return out


variant_metrics = evaluate_by_variant(val_loader)
print('=== Variant-level comparison (prediction vs GT + discriminator) ===')
for v in sorted(variant_metrics.keys()):
    m = variant_metrics[v]
    print(
        f"{v:30s} | PSNR {m['psnr']:.2f} dB | "
        f"D(real) {m['real_score']:.4f} | D(fake) {m['fake_score']:.4f} | "
        f"gap {m['score_gap']:.4f}"
    )

In [ ]:
out_path = Path(CFG['disc_ckpt'])
out_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        'generator_state_dict': G.state_dict(),
        'discriminator_state_dict': D.state_dict(),
        'history': history,
        'variant_metrics': variant_metrics,
        'cfg': CFG,
    },
    out_path,
)

print(f'Saved joint GAN checkpoint bundle: {out_path}')
print(f'Warmup epochs: {CFG["warmup_d_epochs"]} | Joint epochs: {CFG["joint_epochs"]}')
print(f'lambda_adv used: {CFG["lambda_adv"]}')

## Optional Qualitative Full-Volume GIFs (Multiple Sparse Settings)

This section reconstructs an entire volume and exports GIFs for multiple experiment settings.
Run the mesh preview cell below first if you only want to verify the 3D geometry.

- A: half alternating (all5)
- B: 50 uniform (all5)
- C: half alternating (t1n+mask)
- D/E/F: additional sparse variants

Each GIF shows side-by-side:
- Ground truth `t1n`
- Predicted reconstruction
- Absolute error

In [7]:
import imageio.v2 as imageio


def to_uint8(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.uint8)
    y = np.clip((x - lo) / (hi - lo), 0.0, 1.0)
    return (y * 255.0).astype(np.uint8)


def apply_sliding_window_overlap(volume: np.ndarray, window_size: int = 5) -> np.ndarray:
    # Blend neighboring z-slices to reduce flicker and seam artifacts.
    if window_size <= 1:
        return volume.astype(np.float32)

    vol = volume.astype(np.float32)
    h, w, z_count = vol.shape
    radius = max(1, window_size // 2)

    out = np.zeros_like(vol, dtype=np.float32)
    for z in range(z_count):
        z0 = max(0, z - radius)
        z1 = min(z_count, z + radius + 1)

        idx = np.arange(z0, z1)
        # Triangular weights centered at z.
        weights = (radius + 1 - np.abs(idx - z)).astype(np.float32)
        weights = weights / np.sum(weights)

        out[:, :, z] = np.tensordot(vol[:, :, z0:z1], weights, axes=([2], [0]))

    return np.clip(out, 0.0, 1.0).astype(np.float32)


def apply_post_processing_stack(
    volume: np.ndarray,
    median_ksize: int = 3,
    gaussian_sigma: float = 0.7,
    sharpen_alpha: float = 0.10,
) -> np.ndarray:
    # Per-slice denoise + gentle unsharp masking for crisper structures.
    vol = volume.astype(np.float32)
    h, w, z_count = vol.shape
    out = np.zeros_like(vol, dtype=np.float32)

    k = int(max(1, median_ksize))
    if k % 2 == 0:
        k += 1

    for z in range(z_count):
        sl = vol[:, :, z]
        if k > 1:
            sl = cv2.medianBlur(sl, k)
        blur = cv2.GaussianBlur(sl, (0, 0), sigmaX=gaussian_sigma, sigmaY=gaussian_sigma)
        sl = np.clip(sl + sharpen_alpha * (sl - blur), 0.0, 1.0)
        out[:, :, z] = sl

    return out.astype(np.float32)


@torch.no_grad()
def predict_with_tta(
    g_model: nn.Module,
    x_t: torch.Tensor,
    use_tta: bool = True,
    tta_modes: Tuple[str, ...] = ('none', 'hflip', 'vflip', 'hvflip'),
) -> torch.Tensor:
    # x_t: [1, C, D, H, W]
    if not use_tta:
        return g_model(x_t)

    preds = []
    for mode in tta_modes:
        if mode == 'none':
            x_aug = x_t
        elif mode == 'hflip':
            x_aug = torch.flip(x_t, dims=(-1,))
        elif mode == 'vflip':
            x_aug = torch.flip(x_t, dims=(-2,))
        elif mode == 'hvflip':
            x_aug = torch.flip(x_t, dims=(-2, -1))
        else:
            raise ValueError(f'Unsupported TTA mode: {mode}')

        y_aug = g_model(x_aug)
        if mode == 'none':
            y = y_aug
        elif mode == 'hflip':
            y = torch.flip(y_aug, dims=(-1,))
        elif mode == 'vflip':
            y = torch.flip(y_aug, dims=(-2,))
        else:
            y = torch.flip(y_aug, dims=(-2, -1))
        preds.append(y)

    return torch.mean(torch.stack(preds, dim=0), dim=0)


@torch.no_grad()
def reconstruct_single_scale_bidirectional(
    g_model: nn.Module,
    vols: Dict[str, np.ndarray],
    known_mask: np.ndarray,
    input_mode: str,
    img_size: int,
    use_tta: bool = True,
    tta_modes: Tuple[str, ...] = ('none', 'hflip', 'vflip', 'hvflip'),
) -> np.ndarray:
    t1n_gt = vols['t1n'].astype(np.float32)
    h, w, z_count = t1n_gt.shape

    fwd = init_t1n_state_from_known(t1n_gt, known_mask)
    bwd = init_t1n_state_from_known(t1n_gt, known_mask)

    g_model.eval()

    for z in range(z_count):
        z_ids = [z - 4, z - 3, z - 2, z - 1, z - 1]
        depth_items = [
            build_context_slice(vols, fwd, known_mask, zi, img_size, input_mode, use_pred_for_all_channels=True)
            for zi in z_ids
        ]
        x_dc = np.stack(depth_items, axis=0).astype(np.float32)
        x = np.transpose(x_dc, (1, 0, 2, 3)).astype(np.float32)
        x_t = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
        pred = predict_with_tta(g_model, x_t, use_tta=use_tta, tta_modes=tta_modes).squeeze(0).squeeze(0).cpu().numpy()
        fwd[:, :, z] = cv2.resize(pred, (w, h), interpolation=cv2.INTER_LINEAR)

    for z in range(z_count - 1, -1, -1):
        z_ids = [z + 1, z + 2, z + 3, z + 4, z + 4]
        depth_items = [
            build_context_slice(vols, bwd, known_mask, zi, img_size, input_mode, use_pred_for_all_channels=True)
            for zi in z_ids
        ]
        x_dc = np.stack(depth_items, axis=0).astype(np.float32)
        x = np.transpose(x_dc, (1, 0, 2, 3)).astype(np.float32)
        x_t = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
        pred = predict_with_tta(g_model, x_t, use_tta=use_tta, tta_modes=tta_modes).squeeze(0).squeeze(0).cpu().numpy()
        bwd[:, :, z] = cv2.resize(pred, (w, h), interpolation=cv2.INTER_LINEAR)

    return (0.5 * (fwd + bwd)).astype(np.float32)


@torch.no_grad()
def reconstruct_full_volume_bidirectional(
    g_model: nn.Module,
    vols: Dict[str, np.ndarray],
    known_mask: np.ndarray,
    input_mode: str,
    img_size: int,
    use_tta: bool = True,
    tta_modes: Tuple[str, ...] = ('none', 'hflip', 'vflip', 'hvflip'),
    multi_scales: Tuple[float, ...] = (1.0, 0.85),
    scale_weights: Tuple[float, ...] = (0.65, 0.35),
    overlap_window: int = 5,
    enable_post_stack: bool = True,
    post_median_ksize: int = 3,
    post_gaussian_sigma: float = 0.7,
    post_sharpen_alpha: float = 0.10,
    refine_sharpen: float = 0.1,
) -> np.ndarray:
    if len(multi_scales) != len(scale_weights):
        raise ValueError('multi_scales and scale_weights must have the same length')

    weights = np.array(scale_weights, dtype=np.float32)
    if float(np.sum(weights)) <= 0.0:
        raise ValueError('scale_weights must have a positive sum')
    weights = weights / np.sum(weights)

    scale_preds = []
    for scale in multi_scales:
        scaled_img_size = int(round(img_size * float(scale)))
        scaled_img_size = max(64, scaled_img_size)
        # Keep size aligned with UNet down/up sampling.
        scaled_img_size = int(max(64, round(scaled_img_size / 16.0) * 16))

        pred_scale = reconstruct_single_scale_bidirectional(
            g_model=g_model,
            vols=vols,
            known_mask=known_mask,
            input_mode=input_mode,
            img_size=scaled_img_size,
            use_tta=use_tta,
            tta_modes=tta_modes,
        )
        scale_preds.append(pred_scale)

    fused = np.zeros_like(scale_preds[0], dtype=np.float32)
    for w_i, pred_i in zip(weights, scale_preds):
        fused += float(w_i) * pred_i

    # Sliding z-window overlap fusion.
    fused = apply_sliding_window_overlap(fused, window_size=overlap_window)

    # Optional classic sharpen retained for compatibility with downstream cell.
    if refine_sharpen > 0.0:
        blur = cv2.GaussianBlur(fused, (0, 0), sigmaX=0.8, sigmaY=0.8)
        fused = np.clip(fused + refine_sharpen * (fused - blur), 0.0, 1.0)

    if enable_post_stack:
        fused = apply_post_processing_stack(
            fused,
            median_ksize=post_median_ksize,
            gaussian_sigma=post_gaussian_sigma,
            sharpen_alpha=post_sharpen_alpha,
        )

    return fused.astype(np.float32)


def export_volume_qualitative_gif(
    gt_vol: np.ndarray,
    pred_vol: np.ndarray,
    out_path: Path,
    label: str,
    fps: int = 8,
):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    z_count = gt_vol.shape[2]

    # Use independent display ranges so Pred does not get crushed to black.
    gt_lo = float(np.min(gt_vol))
    gt_hi = float(np.max(gt_vol))
    pr_lo = float(np.min(pred_vol))
    pr_hi = float(np.max(pred_vol))

    # Stable error scaling across all slices.
    err_hi = float(np.max(np.abs(gt_vol - pred_vol)) + 1e-8)

    print(
        f"[{label}] GT range=({gt_lo:.6f}, {gt_hi:.6f}) | "
        f"Pred range=({pr_lo:.6f}, {pr_hi:.6f}) | "
        f"Err max={err_hi:.6f}"
    )

    frames = []
    for z in range(z_count):
        gt = gt_vol[:, :, z].astype(np.float32)
        pr = pred_vol[:, :, z].astype(np.float32)
        er = np.abs(gt - pr)

        gt_u8 = np.flipud(to_uint8(gt.T, gt_lo, gt_hi))
        pr_u8 = np.flipud(to_uint8(pr.T, pr_lo, pr_hi))
        er_u8 = np.flipud(to_uint8(er.T, 0.0, err_hi))

        gt_rgb = np.stack([gt_u8, gt_u8, gt_u8], axis=-1)
        pr_rgb = np.stack([pr_u8, pr_u8, pr_u8], axis=-1)
        er_rgb = np.stack([er_u8, er_u8, er_u8], axis=-1)

        frame = np.concatenate([gt_rgb, pr_rgb, er_rgb], axis=1).astype(np.uint8)

        cv2.putText(frame, "GT", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, "Pred", (gt_rgb.shape[1] + 10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, "AbsErr", (gt_rgb.shape[1] + pr_rgb.shape[1] + 10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, f"{label} | z={z}", (10, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        frames.append(frame)

    imageio.mimsave(out_path, frames, fps=fps, loop=0)
    return out_path


# -------------------------
# GIF export is parked below so the mesh can be checked first.
# -------------------------
"""
tp_demo = TIMEPOINTS[0]
vols_demo = load_timepoint_volumes(tp_demo)
gt_demo = vols_demo['t1n'].astype(np.float32)
z_count = gt_demo.shape[2]

# Only requested cases.
variant_cfgs = {
    'A_half_alternating_all5': ('all5', build_sparse_known_masks(z_count)['A_half_alternating_all5']),
    'B_50_uniform_all5': ('all5', build_sparse_known_masks(z_count)['B_50_uniform_all5']),
    'C_half_alternating_t1n_mask': ('t1n_mask', build_sparse_known_masks(z_count)['C_half_alternating_t1n_mask']),
}

qual_dir = Path('assets') / 'recon_media_disc'
qual_dir.mkdir(parents=True, exist_ok=True)

saved_gifs = []
for variant_name, (input_mode, known_mask) in variant_cfgs.items():
    pred_demo = reconstruct_full_volume_bidirectional(
        G,
        vols_demo,
        known_mask=known_mask,
        input_mode=input_mode,
        img_size=CFG['img_size'],
        use_tta=True,
        tta_modes=('none', 'hflip', 'vflip', 'hvflip'),
        multi_scales=(1.0, 0.85),
        scale_weights=(0.65, 0.35),
        overlap_window=5,
        enable_post_stack=True,
        post_median_ksize=3,
        post_gaussian_sigma=0.7,
        post_sharpen_alpha=0.10,
    )

    gif_path = qual_dir / f"{tp_demo.name}_{variant_name}_qual.gif"
    export_volume_qualitative_gif(gt_demo, pred_demo, gif_path, label=variant_name, fps=8)
    saved_gifs.append(gif_path)
    print(f'Saved GIF: {gif_path}')

try:
    from IPython.display import Image, display
    print('\nInline preview of generated qualitative GIFs:')
    for p in saved_gifs:
        print(p)
        display(Image(filename=str(p)))
except Exception as ex:
    print(f'Inline display skipped ({ex}). GIFs are in: {qual_dir}')
"""

'\ntp_demo = TIMEPOINTS[0]\nvols_demo = load_timepoint_volumes(tp_demo)\ngt_demo = vols_demo[\'t1n\'].astype(np.float32)\nz_count = gt_demo.shape[2]\n\n# Only requested cases.\nvariant_cfgs = {\n    \'A_half_alternating_all5\': (\'all5\', build_sparse_known_masks(z_count)[\'A_half_alternating_all5\']),\n    \'B_50_uniform_all5\': (\'all5\', build_sparse_known_masks(z_count)[\'B_50_uniform_all5\']),\n    \'C_half_alternating_t1n_mask\': (\'t1n_mask\', build_sparse_known_masks(z_count)[\'C_half_alternating_t1n_mask\']),\n}\n\nqual_dir = Path(\'assets\') / \'recon_media_disc\'\nqual_dir.mkdir(parents=True, exist_ok=True)\n\nsaved_gifs = []\nfor variant_name, (input_mode, known_mask) in variant_cfgs.items():\n    pred_demo = reconstruct_full_volume_bidirectional(\n        G,\n        vols_demo,\n        known_mask=known_mask,\n        input_mode=input_mode,\n        img_size=CFG[\'img_size\'],\n        use_tta=True,\n        tta_modes=(\'none\', \'hflip\', \'vflip\', \'hvflip\'),\n        

## Final step: build a true 3D brain mesh from CASE A prediction
### Material (vertex color/intensity) is taken from the predicted T1n volume.
### Includes interactive transform controls (rotate/flip/scale/translate) similar to a 3D viewport.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from pathlib import Path
from skimage import measure
from IPython.display import display, clear_output

tp_demo = TIMEPOINTS[0]
vols_demo = load_timepoint_volumes(tp_demo)
gt_demo = vols_demo['t1n'].astype(np.float32)
z_count = gt_demo.shape[2]

variant_cfgs = {
    'A_half_alternating_all5': ('all5', build_sparse_known_masks(z_count)['A_half_alternating_all5']),
    'B_50_uniform_all5': ('all5', build_sparse_known_masks(z_count)['B_50_uniform_all5']),
    'C_half_alternating_t1n_mask': ('t1n_mask', build_sparse_known_masks(z_count)['C_half_alternating_t1n_mask']),
}

# Rebuild CASE A prediction if this kernel was restarted.
if 'pred_case_a' not in globals():
    CASE_A_KEY = 'A_half_alternating_all5'
    input_mode_case_a, known_mask_case_a = variant_cfgs[CASE_A_KEY]
    pred_case_a = reconstruct_full_volume_bidirectional(
        G,
        vols_demo,
        known_mask=known_mask_case_a,
        input_mode=input_mode_case_a,
        img_size=CFG['img_size'],
        use_tta=True,
        tta_modes=('none', 'hflip', 'vflip', 'hvflip'),
        multi_scales=(1.0, 0.85),
        scale_weights=(0.65, 0.35),
        overlap_window=5,
        enable_post_stack=True,
        post_median_ksize=3,
        post_gaussian_sigma=0.7,
        post_sharpen_alpha=0.10,
    ).astype(np.float32)

pred_case_a = np.clip(pred_case_a, 0.0, 1.0)
if 'CASE_A_KEY' not in globals():
    CASE_A_KEY = 'A_half_alternating_all5'
if 'mesh_dir' not in globals():
    mesh_dir = Path('assets') / 'recon_media_disc' / 'meshes'
    mesh_dir.mkdir(parents=True, exist_ok=True)


def rotation_matrix_xyz(rx_deg: float, ry_deg: float, rz_deg: float) -> np.ndarray:
    rx = np.deg2rad(rx_deg)
    ry = np.deg2rad(ry_deg)
    rz = np.deg2rad(rz_deg)

    cx, sx = np.cos(rx), np.sin(rx)
    cy, sy = np.cos(ry), np.sin(ry)
    cz, sz = np.cos(rz), np.sin(rz)

    rx_m = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]], dtype=np.float32)
    ry_m = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]], dtype=np.float32)
    rz_m = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]], dtype=np.float32)
    return (rz_m @ ry_m @ rx_m).astype(np.float32)


def transform_vertices(
    vertices: np.ndarray,
    rx: float, ry: float, rz: float,
    flip_x: bool, flip_y: bool, flip_z: bool,
    sx: float, sy: float, sz: float,
    tx: float, ty: float, tz: float,
) -> np.ndarray:
    v = vertices.copy().astype(np.float32)
    center = v.mean(axis=0, keepdims=True)
    v = v - center

    flip_vec = np.array([
        -1.0 if flip_x else 1.0,
        -1.0 if flip_y else 1.0,
        -1.0 if flip_z else 1.0,
    ], dtype=np.float32)
    v *= flip_vec[None, :]
    v *= np.array([sx, sy, sz], dtype=np.float32)[None, :]
    v = v @ rotation_matrix_xyz(rx, ry, rz).T
    v = v + center + np.array([tx, ty, tz], dtype=np.float32)[None, :]
    return v


def build_mesh_from_iso(volume: np.ndarray, iso_value: float):
    verts_rcz, faces_local, _, sampled_intensity = measure.marching_cubes(volume, level=float(iso_value))

    verts_xyz = np.stack([verts_rcz[:, 2], verts_rcz[:, 1], verts_rcz[:, 0]], axis=1).astype(np.float32)
    faces_local = faces_local.astype(np.int32)
    material_local = sampled_intensity.astype(np.float32)

    mmin = float(material_local.min())
    mmax = float(material_local.max())
    if mmax - mmin < 1e-8:
        material_norm_local = np.zeros_like(material_local, dtype=np.float32)
    else:
        material_norm_local = (material_local - mmin) / (mmax - mmin)

    return verts_xyz, faces_local, material_local, material_norm_local


def section_faces_by_x(vertices_xyz: np.ndarray, faces_arr: np.ndarray, ratio: float, mode: str):
    x_vals = vertices_xyz[:, 0]
    x_min = float(np.min(x_vals))
    x_max = float(np.max(x_vals))
    cut_x = x_min + float(ratio) * (x_max - x_min)

    x_per_face = x_vals[faces_arr]
    if mode == 'left':
        keep = np.all(x_per_face <= cut_x, axis=1)
    elif mode == 'right':
        keep = np.all(x_per_face >= cut_x, axis=1)
    else:
        keep = np.ones(faces_arr.shape[0], dtype=bool)

    return faces_arr[keep], cut_x, x_min, x_max


def make_volume_trace_data(volume: np.ndarray, cut_ratio: float, mode: str, down: int = 3):
    vol_ds = volume[::down, ::down, ::down].astype(np.float32)
    r, c, s = np.indices(vol_ds.shape)

    x = (s * down).astype(np.float32)
    y = (c * down).astype(np.float32)
    z = (r * down).astype(np.float32)

    cut_x = float(cut_ratio * (volume.shape[2] - 1))
    cut_s = cut_x / float(down)

    if mode == 'left':
        keep = s <= cut_s
    elif mode == 'right':
        keep = s >= cut_s
    else:
        keep = np.ones_like(s, dtype=bool)

    val = np.where(keep, vol_ds, 0.0)
    return x.ravel(), y.ravel(), z.ravel(), val.ravel(), cut_x


non_zero_vals = pred_case_a[pred_case_a > 0]
if non_zero_vals.size == 0:
    raise RuntimeError('pred_case_a is empty; cannot build mesh.')

iso_min = float(np.percentile(non_zero_vals, 55))
iso_max = float(np.percentile(non_zero_vals, 95))
iso_default = float(0.23)
base_vertices, faces, material, material_norm = build_mesh_from_iso(pred_case_a, iso_default)
transformed_vertices = base_vertices.copy()
current_state = {}

iso_w = widgets.FloatSlider(
    value=iso_default,
    min=iso_min,
    max=iso_max,
    step=max((iso_max - iso_min) / 250.0, 1e-4),
    description='ISO',
    readout_format='.4f',
)
section_w = widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.005, description='Section', readout_format='.3f')
section_mode_w = widgets.Dropdown(
    options=[
        ('Show left part (x <= cut)', 'left'),
        ('Show right part (x >= cut)', 'right'),
        ('Show full mesh', 'full'),
    ],
    value='left',
    description='Cut Mode',
)
show_fill_w = widgets.Checkbox(value=True, description='Filled interior')
volume_down_w = widgets.IntSlider(value=2, min=2, max=6, step=1, description='Vol DS')
fill_opacity_w = widgets.FloatSlider(value=0.2, min=0.05, max=0.50, step=0.01, description='Fill Opacity')
fill_surface_w = widgets.IntSlider(value=40, min=8, max=40, step=1, description='Fill Surfaces')
mesh_opacity_w = widgets.FloatSlider(value=0.8, min=0.20, max=1.00, step=0.02, description='Mesh Opacity')

rx_w = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='Rot X')
ry_w = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='Rot Y')
rz_w = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='Rot Z')
flipx_w = widgets.Checkbox(value=False, description='Flip X')
flipy_w = widgets.Checkbox(value=False, description='Flip Y')
flipz_w = widgets.Checkbox(value=False, description='Flip Z')
sx_w = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.01, description='Scale X')
sy_w = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.01, description='Scale Y')
sz_w = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.01, description='Scale Z')
tx_w = widgets.FloatSlider(value=0.0, min=-120.0, max=120.0, step=1.0, description='Move X')
ty_w = widgets.FloatSlider(value=0.0, min=-120.0, max=120.0, step=1.0, description='Move Y')
tz_w = widgets.FloatSlider(value=0.0, min=-120.0, max=120.0, step=1.0, description='Move Z')

section_info = widgets.HTML(value='')
out = widgets.Output(layout=widgets.Layout(border='1px solid #dddddd'))


def build_figure():
    global transformed_vertices, current_state

    transformed_vertices = transform_vertices(
        base_vertices,
        rx_w.value, ry_w.value, rz_w.value,
        flipx_w.value, flipy_w.value, flipz_w.value,
        sx_w.value, sy_w.value, sz_w.value,
        tx_w.value, ty_w.value, tz_w.value,
    )

    faces_sec, cut_x, xmin, xmax = section_faces_by_x(
        transformed_vertices,
        faces,
        ratio=section_w.value,
        mode=section_mode_w.value,
    )

    section_info.value = (
        f'<b>Cut x:</b> {cut_x:.2f} &nbsp; '
        f'<b>range:</b> [{xmin:.2f}, {xmax:.2f}] &nbsp; '
        f'<b>mesh faces shown:</b> {faces_sec.shape[0]:,}'
    )

    xv, yv, zv, vv, _ = make_volume_trace_data(
        pred_case_a,
        cut_ratio=section_w.value,
        mode=section_mode_w.value,
        down=int(volume_down_w.value),
    )

    vv_pos = vv[vv > 0]
    if vv_pos.size:
        fill_isomin = max(0.02, float(np.percentile(vv_pos, 5)))
        fill_isomax = max(fill_isomin + 0.05, float(np.percentile(vv_pos, 98)))
    else:
        fill_isomin = 0.02
        fill_isomax = 0.95

    current_state = {
        'transformed_vertices': transformed_vertices,
        'faces_sec': faces_sec,
        'cut_x': cut_x,
        'xmin': xmin,
        'xmax': xmax,
    }

    mesh_trace = go.Mesh3d(
        x=transformed_vertices[:, 0].tolist(),
        y=transformed_vertices[:, 1].tolist(),
        z=transformed_vertices[:, 2].tolist(),
        i=faces_sec[:, 0].tolist() if faces_sec.size else [],
        j=faces_sec[:, 1].tolist() if faces_sec.size else [],
        k=faces_sec[:, 2].tolist() if faces_sec.size else [],
        intensity=material_norm.tolist(),
        colorscale='Greys',
        cmin=0.0,
        cmax=1.0,
        flatshading=False,
        opacity=float(mesh_opacity_w.value),
        showscale=True,
        colorbar={'title': 'T1n material'},
        lighting={'ambient': 0.45, 'diffuse': 0.75, 'specular': 0.20, 'roughness': 0.55},
        lightposition={'x': 300, 'y': 300, 'z': 600},
        name='CASE A Mesh',
    )

    volume_trace = go.Volume(
        x=xv.tolist(),
        y=yv.tolist(),
        z=zv.tolist(),
        value=vv.tolist(),
        isomin=fill_isomin,
        isomax=fill_isomax,
        opacity=float(fill_opacity_w.value),
        surface_count=int(fill_surface_w.value),
        colorscale='Greys',
        opacityscale=[[0.0, 0.0], [0.03, 0.02], [0.08, 0.08], [0.20, 0.18], [0.45, 0.38], [1.0, 0.70]],
        caps={'x_show': False, 'y_show': False, 'z_show': False},
        showscale=False,
        visible=bool(show_fill_w.value and vv_pos.size > 0),
        name='Filled interior',
    )

    fig_local = go.Figure(data=[mesh_trace, volume_trace])
    fig_local.update_layout(
        width=980,
        height=720,
        title='CASE A Mesh | ISO + X-Section + Transform',
        scene={
            'xaxis_title': 'X',
            'yaxis_title': 'Y',
            'zaxis_title': 'Z',
            'aspectmode': 'data',
            'dragmode': 'orbit',
        },
        margin={'l': 0, 'r': 0, 't': 50, 'b': 0},
    )
    current_state['fig'] = fig_local
    return fig_local


def render_scene(_=None):
    global fig
    fig = build_figure()
    with out:
        clear_output(wait=True)
        display(fig)


def on_iso_change(change):
    global base_vertices, faces, material, material_norm
    base_vertices, faces, material, material_norm = build_mesh_from_iso(pred_case_a, float(change['new']))
    render_scene()


def reset_transforms(_):
    rx_w.value = 0.0
    ry_w.value = 0.0
    rz_w.value = 0.0
    flipx_w.value = False
    flipy_w.value = False
    flipz_w.value = False
    sx_w.value = 1.0
    sy_w.value = 1.0
    sz_w.value = 1.0
    tx_w.value = 0.0
    ty_w.value = 0.0
    tz_w.value = 0.0


def export_transformed_mesh(_):
    state = current_state if current_state else {}
    faces_sec = state.get('faces_sec', faces)
    cut_x = state.get('cut_x', 0.0)

    out_path = mesh_dir / f"{CASE_A_KEY}_mesh_iso_{iso_w.value:.4f}_section_{section_mode_w.value}.npz"
    np.savez_compressed(
        out_path,
        vertices=transformed_vertices.astype(np.float32),
        faces=faces_sec.astype(np.int32),
        full_faces=faces.astype(np.int32),
        material=material.astype(np.float32),
        material_norm=material_norm.astype(np.float32),
        iso=np.array([iso_w.value], dtype=np.float32),
        cut_x=np.array([cut_x], dtype=np.float32),
        section_ratio=np.array([section_w.value], dtype=np.float32),
        section_mode=np.array([section_mode_w.value]),
        params=np.array([
            rx_w.value, ry_w.value, rz_w.value,
            float(flipx_w.value), float(flipy_w.value), float(flipz_w.value),
            sx_w.value, sy_w.value, sz_w.value,
            tx_w.value, ty_w.value, tz_w.value,
        ], dtype=np.float32),
    )
    print(f'Exported mesh: {out_path}')


iso_w.observe(on_iso_change, names='value')
for w in [
    section_w, section_mode_w,
    rx_w, ry_w, rz_w,
    flipx_w, flipy_w, flipz_w,
    sx_w, sy_w, sz_w,
    tx_w, ty_w, tz_w,
    show_fill_w, volume_down_w,
    fill_opacity_w, fill_surface_w, mesh_opacity_w,
]:
    w.observe(render_scene, names='value')

reset_btn = widgets.Button(description='Reset Transforms', button_style='warning')
reset_btn.on_click(reset_transforms)
export_btn = widgets.Button(description='Export Mesh', button_style='success')
export_btn.on_click(export_transformed_mesh)

controls = widgets.VBox([
    widgets.HTML('<b>ISO + X-Section</b>'),
    iso_w,
    widgets.HBox([section_w, section_mode_w]),
    widgets.HBox([show_fill_w, volume_down_w]),
    widgets.HBox([fill_opacity_w, fill_surface_w]),
    mesh_opacity_w,
    section_info,
    widgets.HTML('<hr><b>Transform</b>'),
    rx_w, ry_w, rz_w,
    widgets.HBox([flipx_w, flipy_w, flipz_w]),
    sx_w, sy_w, sz_w,
    tx_w, ty_w, tz_w,
    widgets.HBox([reset_btn, export_btn]),
])

display(widgets.HBox([controls, out]))
render_scene()
